In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import copy
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, normalize
import scipy.cluster.hierarchy as shc
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV
from sklearn import metrics
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorboardX import SummaryWriter
import os
import time
from torcheval.metrics import R2Score
from sklearn.metrics import r2_score

%matplotlib inline
pd.set_option('display.max_columns', None)

In [2]:
def swap_columns(df, col1, col2):
    col_list = list(df.columns)
    x, y = col_list.index(col1), col_list.index(col2)
    col_list[y], col_list[x] = col_list[x], col_list[y]
    df = df[col_list]
    return df

# data

In [3]:
# loading the imputed dataset (no null values)
test_clean_df = pd.read_csv('../data/test_factor_loadings_5.csv')
test_clean_df_ss = test_clean_df['subject_session'].values
test_clean_df_ss = [i.split('_')[0] + '_PRE' for i in test_clean_df_ss]
test_clean_df['subject_session'] = test_clean_df_ss

retest_clean_df = pd.read_csv('../data/retest_factor_loadings_5.csv')
retest_clean_df_ss = retest_clean_df['subject_session'].values
retest_clean_df_ss = [i.split('_')[0] + '_POST' for i in retest_clean_df_ss]
retest_clean_df['subject_session'] = retest_clean_df_ss


In [4]:
# loading profiles
demographic_df = pd.read_csv('../data/demographic.csv')
demographic_df.rename(columns={'subject': 'subject_session'}, inplace=True)
cognitive_df = pd.read_csv('../data/cognitive.csv')
cognitive_df.rename(columns={'subject':'subject_session'}, inplace=True)

In [5]:
cognitive_clean_df = cognitive_df.dropna().reset_index().drop(columns='index')

In [6]:
cognitive_neat_ones_test = cognitive_clean_df['subject_session'].values
cognitive_neat_ones_retest = [i.split('_')[0] + '_POST' for i in cognitive_neat_ones_test]

demographic_ones_test = demographic_df['subject_session'].values
demographic_ones_retest = [i.split('_')[0] + '_POST' for i in demographic_ones_test]

In [7]:
pre_cog_merge = pd.merge(cognitive_clean_df, test_clean_df, on='subject_session')
pre_all_merge = pd.merge(demographic_df, pre_cog_merge, on='subject_session')

In [8]:
cognitive_clean_df['subject_session'] = cognitive_neat_ones_retest
demographic_df['subject_session'] = demographic_ones_retest

post_cog_merge = pd.merge(cognitive_clean_df, retest_clean_df, on='subject_session')
post_all_merge = pd.merge(demographic_df, post_cog_merge, on='subject_session')

In [9]:
final_df = pd.concat([pre_all_merge, post_all_merge], axis=0).reset_index().drop(columns='index')
final_df['gender'] = [0 if i=='F' else 1 for i in final_df['gender']]
final_df = swap_columns(final_df, 'age', 'ageGroup')
final_df

,subject_session,ageGroup,age,gender,musicAllYears,musicFormal,cog_wm_omissions,cog_wm_rt,cog_flex_totalperf,cog_gonogo_errors,cog_gonogo_rt,wasiMatrixT,wasiVocabT,wasiFSIQ2,0,1,2,3,4
0,A03_PRE,Age_22_29,22,0,4.0,0.0,44.0,43.0,58.0,40.0,52.0,50.0,80.0,126.0,-0.880371,2.302277,-0.999466,-0.305742,2.233433
1,A04_PRE,Age_22_29,22,0,0.5,0.0,36.0,41.0,61.0,47.0,43.0,61.0,80.0,135.0,-0.781987,-0.450973,-0.962141,0.249724,-0.163783
2,A07_PRE,Age_22_29,22,1,0.0,0.0,44.0,43.0,46.0,50.0,49.0,42.0,40.0,84.0,-0.389527,1.092750,-0.969593,0.520481,-0.026118
3,A08_PRE,Age_18_21,18,1,4.0,0.0,48.0,39.0,48.0,50.0,52.0,68.0,80.0,142.0,-0.945165,0.383428,-0.829711,0.062983,2.192393
4,A09_PRE,Age_22_29,25,1,3.0,0.0,50.0,41.0,38.0,50.0,44.0,55.0,54.0,108.0,-1.048721,-0.381687,1.760557,0.173195,0.121621
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,F03_POST,Age_55_87,71,0,0.0,0.0,73.0,59.0,74.0,50.0,54.0,79.0,80.0,142.0,-0.152185,-0.745060,0.672865,-0.541989,-0.780706
115,F07_POST,Age_55_87,87,1,0.0,0.0,76.0,60.0,79.0,50.0,44.0,61.0,60.0,118.0,0.036425,-0.195460,-0.397836,0.189870,-0.570293
116,F08_POST,Age_55_87,72,1,3.0,0.0,69.0,53.0,63.0,50.0,43.0,61.0,72.0,129.0,-0.788878,-0.519137,-0.752896,0.064376,-0.049678
117,F11_POST,Age_55_87,80,0,1.0,0.0,73.0,57.0,62.0,50.0,65.0,66.0,73.0,134.0,-0.159350,1.756886,-0.739771,-0.268402,-0.385162


In [10]:
final_df.to_csv('../data/deep_factor_test_retest_imputed.csv', index_label=False)

In [11]:
final_df_all_standardized = copy.deepcopy(final_df)
final_df_all_standardized.iloc[:,2:] = StandardScaler().fit_transform(final_df_all_standardized.iloc[:,2:])
final_df_all_standardized.to_csv('../data/deep_factor_test_retest_imputed_all_standardized.csv', index_label=False)

/Users/matin/miniconda3/envs/phd_codes/lib/python3.11/site-packages/sklearn/utils/validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Users/matin/miniconda3/envs/phd_codes/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/matin/miniconda3/envs/phd_codes/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
/Users/matin/miniconda3/envs/phd_codes/lib/python3.11/site-packages/sklearn/utils/validation.py:767: FutureWarning: is_sparse is deprec

In [12]:
final_df_measures_standardized = copy.deepcopy(final_df)
final_df_measures_standardized.iloc[:,14:] = StandardScaler().fit_transform(final_df_measures_standardized.iloc[:,14:])
final_df_measures_standardized.to_csv('../data/deep_factor_test_retest_imputed_measures_standardized.csv', index_label=False)


/Users/matin/miniconda3/envs/phd_codes/lib/python3.11/site-packages/sklearn/utils/validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
/Users/matin/miniconda3/envs/phd_codes/lib/python3.11/site-packages/sklearn/utils/validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/Users/matin/miniconda3/envs/phd_codes/lib/python3.11/site-packages/sklearn/utils/validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
/Users/matin/miniconda3/envs/phd_codes/lib/python3.11/site-packages/sklearn/utils/validation.py:767: FutureWarning: is_sparse is deprec